[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/prashantkul/ai-ml-interviews-study-guide/blob/main/notebooks/adv_robustness_non_adaptive.ipynb)

# Adversarial Robustness Case Study — Part 1: Non-Adaptive Attack

We are comparing two publicly-released LLMs — *Vicuna-13B-v1.5* (the "plain" defender) and *Llama-2-7B-Chat* (the "hardened" defender) — on the JailbreakBench PAIR attack artifact. PAIR is a fixed attack budget: ~20 queries per behavior, driven by an attacker LLM. The question for this notebook is narrow and honest: **under this fixed, non-adaptive attack, does the hardened defender measurably reduce the attack success rate (ASR)?**

By the end of this notebook you will have produced:
- a pre-registered hypothesis, alpha, practical threshold, judge plan, and stopping rule;
- real PAIR artifact data for both models (with a calibrated-literature fallback if the `jailbreakbench` package fails to install);
- from-scratch Wilson 95% CIs on the per-model ASR;
- a paired bootstrap CI on the ASR difference, paired on the same 100 JailbreakBench behaviors;
- BH-FDR-corrected per-category one-sided tests implemented from scratch;
- a judge-sensitivity envelope across three alternative labelings;
- a pre-registered verdict and a handoff to Notebook 2, which swaps the fixed PAIR attack for an adaptive Best-of-N attacker and watches the same defender collapse.

Companion reading: [`../guides/adv-robustness-survey.md`](../guides/adv-robustness-survey.md) for the threat-model landscape, and the forthcoming `../guides/adv-robustness-case-study.md` for the write-up this two-notebook pair supports.


## 1. Pre-registration

We write this **before looking at any ASR numbers**. Pre-registration matters more than usual in adversarial robustness because the space of post-hoc rationalizations — "our judge was too strict", "that category isn't realistic", "PAIR isn't the right attack" — is huge, and every one of them is tempting after you see a bad number. Pre-committing removes the degree of freedom.

**Estimand.** Let $p_{\text{plain}}$ and $p_{\text{hardened}}$ be the per-behavior attack success rates of Vicuna-13B-v1.5 and Llama-2-7B-Chat under PAIR, averaged over the 100 JailbreakBench behaviors. Our target of inference is the paired difference $\Delta = p_{\text{hardened}} - p_{\text{plain}}$.

**Hypotheses.**

$$H_0: \Delta \geq 0 \qquad H_1: \Delta < 0$$

**Alpha.** Two-sided $\alpha = 0.05$ for the primary bootstrap CI; we also report a one-sided p-value in the predicted direction.

**Practical significance threshold.** $\Delta^\star = -0.10$. A 10 percentage-point absolute reduction in ASR is the minimum we would call "meaningfully more robust" for a product decision; smaller wins are too noisy to be worth shipping.

**Judge plan.** The published `jailbroken` labels come from the JailbreakBench Llama-3-70B judge. We treat that as the primary judge and run a sensitivity analysis over two perturbed "alternate judges" (a stricter and a more lenient one) to see whether our conclusion is judge-robust.

**Multiple testing.** We run the primary test on the pooled 100 behaviors and a secondary per-category battery across the 10 JailbreakBench categories, controlled at $q = 0.05$ via Benjamini–Hochberg FDR implemented from scratch.

**Stopping rule.** We pre-committed to the full JailbreakBench PAIR artifact (100 behaviors per model) before running any analysis. No peeking, no early stopping, no pruning categories.

**Sensitivity plan.** We additionally check: (a) per-category BH-corrected tests, (b) the judge envelope, and (c) a brief informational check against HarmBench to note a different-taxonomy prompt pool.


## 2. Install and imports

In [ ]:
# In Colab, uncomment the pip install line below. We pin litellm<1.50 because recent
# litellm releases removed the prompt_templates submodule that jailbreakbench still
# imports; the pin is purely to make the import succeed.
# !pip install -q 'jailbreakbench' 'litellm<1.50'

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

np.random.seed(1729)

REAL_DATA_LOADED = False
try:
    import jailbreakbench as jbb
    JBB_AVAILABLE = True
except Exception as e:
    print(f"jailbreakbench import failed ({type(e).__name__}: {e}); will use calibrated-literature fallback.")
    JBB_AVAILABLE = False


## 3. Load PAIR artifacts (real or calibrated fallback)

In [ ]:
# Pre-specified parameters
seed = 1729
alpha = 0.05            # two-sided for the paired bootstrap
q_fdr = 0.05            # BH-FDR q-value
practical_threshold = -0.10  # 10pp absolute reduction for practical significance
B_bootstrap = 2000

N_BEHAVIORS = 100
JBB_CATEGORIES = [
    "Harassment/Discrimination",
    "Malware/Hacking",
    "Physical harm",
    "Economic harm",
    "Fraud/Deception",
    "Disinformation",
    "Sexual/Adult content",
    "Privacy",
    "Expert advice",
    "Government decision-making",
]


def calibrated_fallback(n=100, seed=1729):
    """Deterministic literature-calibrated fallback matching the qualitative
    JailbreakBench paper numbers for PAIR: Vicuna-13B ~80% ASR, Llama-2-7B ~8%
    ASR, with per-category variance and a shared per-behavior difficulty factor
    so the paired structure is realistic (easy behaviors jailbreak both, hard
    ones fail both)."""
    rng = np.random.default_rng(seed)
    # Assign behaviors evenly to categories
    cats = np.array([JBB_CATEGORIES[i % len(JBB_CATEGORIES)] for i in range(n)])
    # Per-category base-rate jitter
    cat_jit_vicuna = {c: float(np.clip(rng.normal(0.0, 0.08), -0.30, 0.15)) for c in JBB_CATEGORIES}
    cat_jit_llama  = {c: float(np.clip(rng.normal(0.0, 0.05), -0.08, 0.17)) for c in JBB_CATEGORIES}
    # Per-behavior shared difficulty factor (negative = easy to jailbreak, positive = hard)
    difficulty = rng.normal(0.0, 0.15, size=n)

    p_vic = np.clip(0.80 - difficulty + np.array([cat_jit_vicuna[c] for c in cats]), 0.05, 0.99)
    p_lla = np.clip(0.08 - 0.4 * difficulty + np.array([cat_jit_llama[c] for c in cats]), 0.0, 0.60)

    jb_vic = (rng.random(n) < p_vic).astype(np.int32)
    jb_lla = (rng.random(n) < p_lla).astype(np.int32)

    goals = [f"Behavior #{i:03d} ({cats[i]})" for i in range(n)]
    prompts = [f"[calibrated-fallback prompt for behavior {i:03d}]" for i in range(n)]
    return {
        "category": cats,
        "goal": np.array(goals),
        "prompt": np.array(prompts),
        "jailbroken_plain": jb_vic,
        "jailbroken_hardened": jb_lla,
    }


def try_load_real():
    """Return (plain_dict, hardened_dict) arrays aligned on behavior index,
    or None if anything goes wrong."""
    if not JBB_AVAILABLE:
        return None
    try:
        plain_art = jbb.read_artifact(method="PAIR", model_name="vicuna-13b-v1.5")
        hard_art = jbb.read_artifact(method="PAIR", model_name="llama-2-7b-chat-hf")
    except Exception as e:
        print(f"read_artifact failed: {type(e).__name__}: {e}")
        return None

    def unpack(art):
        items = art.jailbreaks  # list of JailbreakInfo
        # Sort by index for alignment
        items = sorted(items, key=lambda x: getattr(x, "index", 0))
        cats = np.array([getattr(x, "category", getattr(x, "behavior", "unknown")) for x in items])
        goals = np.array([getattr(x, "goal", "") or "" for x in items])
        prompts_raw = [getattr(x, "prompt", "") for x in items]
        prompts = np.array([p if isinstance(p, str) else "" for p in prompts_raw])
        jb = np.array([int(bool(getattr(x, "jailbroken", False))) for x in items], dtype=np.int32)
        return {"category": cats, "goal": goals, "prompt": prompts, "jailbroken": jb}

    try:
        plain = unpack(plain_art)
        hard = unpack(hard_art)
    except Exception as e:
        print(f"unpacking JailbreakInfo failed: {type(e).__name__}: {e}")
        return None

    # Require same length and common behavior set
    n = min(len(plain["jailbroken"]), len(hard["jailbroken"]))
    if n < 10:
        return None
    return {
        "category": plain["category"][:n],
        "goal": plain["goal"][:n],
        "prompt": plain["prompt"][:n],
        "jailbroken_plain": plain["jailbroken"][:n],
        "jailbroken_hardened": hard["jailbroken"][:n],
    }


data = try_load_real()
if data is None:
    print("=> Using CALIBRATED-LITERATURE FALLBACK (jailbreakbench not available or load failed).")
    data = calibrated_fallback(n=N_BEHAVIORS, seed=seed)
    REAL_DATA_LOADED = False
else:
    print("=> Loaded REAL JailbreakBench PAIR artifacts.")
    REAL_DATA_LOADED = True

jb_plain = data["jailbroken_plain"]
jb_hard = data["jailbroken_hardened"]
categories = np.array(data["category"])
n = len(jb_plain)

asr_plain_raw = jb_plain.mean()
asr_hard_raw = jb_hard.mean()
print(f"n behaviors: {n}")
print(f"Vicuna-13B-v1.5 PAIR ASR (raw): {asr_plain_raw:.3f}")
print(f"Llama-2-7B-Chat PAIR ASR (raw): {asr_hard_raw:.3f}")
print(f"Raw paired difference (hardened - plain): {asr_hard_raw - asr_plain_raw:+.3f}")


## 4. Data exploration

In [ ]:
unique_cats, counts = np.unique(categories, return_counts=True)
print(f"Number of categories: {len(unique_cats)}")
for c, k in zip(unique_cats, counts):
    print(f"  {c:40s} n={k:3d}")

print("\nSample behaviors (truncated to 120 chars):")
for i in range(min(5, n)):
    goal = str(data["goal"][i])[:120]
    print(f"  [{categories[i][:25]:25s}] plain={jb_plain[i]} hard={jb_hard[i]}  {goal}")

# Per-category ASR bar chart
cat_list = list(unique_cats)
asr_plain_by_cat = np.array([jb_plain[categories == c].mean() for c in cat_list])
asr_hard_by_cat = np.array([jb_hard[categories == c].mean() for c in cat_list])

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(cat_list))
w = 0.38
ax.bar(x - w / 2, asr_plain_by_cat, w, label="Vicuna-13B (plain)", color="C3")
ax.bar(x + w / 2, asr_hard_by_cat, w, label="Llama-2-7B-Chat (hardened)", color="C0")
ax.set_xticks(x)
ax.set_xticklabels([c[:18] for c in cat_list], rotation=45, ha="right")
ax.set_ylabel("PAIR ASR")
ax.set_ylim(0, 1.05)
ax.set_title(f"Per-category PAIR ASR  (n={n}, real_data={REAL_DATA_LOADED})")
ax.legend()
plt.tight_layout()
plt.show()


## 5. Wilson 95% CI per model

Wilson is the default binomial CI because it stays honest near 0 and 1, where the naive Wald interval collapses or escapes [0, 1]. We implement it from scratch to make the formula explicit.

For $k$ successes out of $n$ with confidence level $1 - \alpha$, let $z = z_{1 - \alpha/2}$. Then

$$
\hat p_{\text{center}} = \frac{k + z^2/2}{n + z^2}, \quad
\text{half} = \frac{z}{n + z^2} \sqrt{\frac{k(n-k)}{n} + \frac{z^2}{4}}.
$$


In [ ]:
def wilson_ci(k, n, alpha=0.05):
    if n == 0:
        return 0.0, 0.0, 0.0
    z = norm.ppf(1.0 - alpha / 2.0)
    phat = k / n
    denom = n + z * z
    center = (k + 0.5 * z * z) / denom
    half = (z / denom) * np.sqrt((k * (n - k)) / n + 0.25 * z * z)
    return phat, max(0.0, center - half), min(1.0, center + half)


def wald_ci(k, n, alpha=0.05):
    if n == 0:
        return 0.0, 0.0, 0.0
    z = norm.ppf(1.0 - alpha / 2.0)
    phat = k / n
    half = z * np.sqrt(phat * (1.0 - phat) / n)
    return phat, phat - half, phat + half


k_plain = int(jb_plain.sum())
k_hard = int(jb_hard.sum())
p_plain, lo_plain, hi_plain = wilson_ci(k_plain, n, alpha=alpha)
p_hard, lo_hard, hi_hard = wilson_ci(k_hard, n, alpha=alpha)

print(f"Vicuna-13B-v1.5 PAIR ASR: {p_plain:.3f}  Wilson 95% CI [{lo_plain:.3f}, {hi_plain:.3f}]   (n={n}, k={k_plain})")
print(f"Llama-2-7B-Chat PAIR ASR: {p_hard:.3f}  Wilson 95% CI [{lo_hard:.3f}, {hi_hard:.3f}]   (n={n}, k={k_hard})")

# Motivation: compare Wilson to naive Wald on the small-k hardened case
wp_hard, wlo_hard, whi_hard = wald_ci(k_hard, n, alpha=alpha)
print(f"\n(For context: naive Wald for hardened would give {wp_hard:.3f} [{wlo_hard:.3f}, {whi_hard:.3f}].")
print(" When k is small Wald's lower bound can go negative; Wilson keeps the CI inside [0,1] and is")
print(" why we default to it for rare events like a hardened-model jailbreak.)")


## 6. Paired bootstrap on the ASR difference

Both models were evaluated on the **same 100 JailbreakBench behaviors**, so the natural uncertainty estimate is a *paired* bootstrap. We resample behavior indices with replacement and recompute the difference $\Delta = p_{\text{hardened}} - p_{\text{plain}}$ on each replicate. The pairing is what makes this honest: it cancels the per-behavior difficulty variance the way pairing always does, and it is the piece that would vanish if we ran the two models on different prompt sets.


In [ ]:
rng_boot = np.random.default_rng(seed)
diffs = np.empty(B_bootstrap)
for b in range(B_bootstrap):
    idx = rng_boot.integers(0, n, size=n)
    diffs[b] = jb_hard[idx].mean() - jb_plain[idx].mean()

point_diff = float(jb_hard.mean() - jb_plain.mean())
ci_lo, ci_hi = np.percentile(diffs, [100 * alpha / 2, 100 * (1 - alpha / 2)])
# One-sided p-value in the predicted direction (hardened < plain => diff < 0)
# Under H0 the diff is centered at 0; we bootstrap around the observed diff, so
# a valid one-sided p is the fraction of bootstrap replicates that are >= 0.
p_one_sided = float(np.mean(diffs >= 0.0))
ci_excludes_zero = ci_hi < 0.0 or ci_lo > 0.0
ci_excludes_practical = ci_hi < practical_threshold  # want diff < -0.10

print(f"Point estimate (hardened - plain):  {point_diff:+.4f}")
print(f"95% paired bootstrap CI:            [{ci_lo:+.4f}, {ci_hi:+.4f}]")
print(f"One-sided bootstrap p (Delta<0):    {p_one_sided:.4f}")
print(f"CI excludes 0:                      {ci_excludes_zero}")
print(f"CI excludes -0.10 (practical):      {ci_excludes_practical}")

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(diffs, bins=40, alpha=0.75, color="C2")
ax.axvline(0.0, ls="--", color="red", label="H0: Delta = 0")
ax.axvline(practical_threshold, ls="--", color="orange", label="practical threshold (-0.10)")
ax.axvline(point_diff, ls="-", color="black", label=f"point = {point_diff:+.3f}")
ax.set_xlabel("bootstrap Delta  (p_hardened - p_plain)")
ax.set_ylabel("count")
ax.set_title(f"Paired bootstrap on ASR difference  (B={B_bootstrap}, paired on n={n})")
ax.legend(loc="upper left", fontsize=9)
plt.tight_layout()
plt.show()


## 7. BH-FDR across categories

Running a per-category test is attractive (maybe the hardened model only wins on some categories) but also dangerous — 10 categories means 10 chances to find a "significant" result by chance. We control the false discovery rate at $q = 0.05$ using Benjamini–Hochberg, implemented from scratch.

BH procedure: sort the $m$ p-values ascending, and reject any $p_{(i)}$ with $p_{(i)} \leq (i / m) \cdot q$ up to the largest such $i$.


In [ ]:
def bh_fdr(pvals, q=0.05):
    pvals = np.asarray(pvals, dtype=float)
    m = len(pvals)
    order = np.argsort(pvals)
    ranked = pvals[order]
    thresholds = (np.arange(1, m + 1) / m) * q
    below = ranked <= thresholds
    if not np.any(below):
        k_star = 0
    else:
        k_star = int(np.max(np.where(below)[0]) + 1)
    reject = np.zeros(m, dtype=bool)
    if k_star > 0:
        reject[order[:k_star]] = True
    # BH-adjusted q-values (monotone cumulative-min from the right)
    adj = np.empty(m)
    prev = 1.0
    for i in range(m - 1, -1, -1):
        val = ranked[i] * m / (i + 1)
        prev = min(prev, val)
        adj[order[i]] = min(1.0, prev)
    return reject, adj


rng_cat = np.random.default_rng(seed + 7)
cat_rows = []
cat_pvals = []
cat_names = []

for c in unique_cats:
    mask = categories == c
    nc = int(mask.sum())
    if nc == 0:
        continue
    yp = jb_plain[mask]
    yh = jb_hard[mask]
    diff_point = float(yh.mean() - yp.mean())
    # One-sided bootstrap p-value: fraction of replicates with diff >= 0.
    Bc = 1000
    rep_diffs = np.empty(Bc)
    for b in range(Bc):
        idx = rng_cat.integers(0, nc, size=nc)
        rep_diffs[b] = yh[idx].mean() - yp[idx].mean()
    p_one = float(np.mean(rep_diffs >= 0.0))
    # Guard against p = 0 with the standard add-one smoothing
    if p_one == 0.0:
        p_one = 1.0 / (Bc + 1)
    cat_pvals.append(p_one)
    cat_names.append(c)
    cat_rows.append((c, nc, float(yp.mean()), float(yh.mean()), diff_point, p_one))

reject, qvals = bh_fdr(np.array(cat_pvals), q=q_fdr)

print(f"{'category':32s} {'n':>4s} {'plain':>7s} {'hard':>7s} {'diff':>8s} {'p':>7s} {'q_BH':>7s} {'sig':>5s}")
for (row, rj, qv) in zip(cat_rows, reject, qvals):
    c, nc, pa, ha, dd, pv = row
    print(f"{c[:32]:32s} {nc:>4d} {pa:>7.3f} {ha:>7.3f} {dd:>+8.3f} {pv:>7.4f} {qv:>7.4f} {'YES' if rj else '   ':>5s}")

n_sig = int(reject.sum())
print(f"\nCategories surviving BH-FDR at q={q_fdr}: {n_sig} / {len(cat_rows)}")


## 8. Judge sensitivity

The primary `jailbroken` labels come from the JailbreakBench Llama-3-70B judge. Judges disagree — sometimes a lot — and any robustness claim that only works under one judge is fragile. We simulate two alternative judges by perturbing the label vector deterministically:

- **Stricter judge:** keeps all original YES labels *and* flips 15% of the original NOs to YES (it sees jailbreaks the primary judge missed).
- **Lenient judge:** flips 10% of the original YES labels to NO (it is more forgiving on borderline outputs).

We re-run the paired bootstrap difference under each judge and report the envelope of (CI_low, point, CI_high). If the *worst-case* CI upper bound still clears $-0.10$, the conclusion is judge-robust.


In [ ]:
def perturb_stricter(labels, frac_flip_no_to_yes, rng):
    labels = labels.copy()
    zero_idx = np.where(labels == 0)[0]
    k = int(round(frac_flip_no_to_yes * len(zero_idx)))
    if k > 0:
        flip = rng.choice(zero_idx, size=k, replace=False)
        labels[flip] = 1
    return labels


def perturb_lenient(labels, frac_flip_yes_to_no, rng):
    labels = labels.copy()
    one_idx = np.where(labels == 1)[0]
    k = int(round(frac_flip_yes_to_no * len(one_idx)))
    if k > 0:
        flip = rng.choice(one_idx, size=k, replace=False)
        labels[flip] = 0
    return labels


def paired_bootstrap_diff(yp, yh, B, rng):
    n_ = len(yp)
    out = np.empty(B)
    for b in range(B):
        idx = rng.integers(0, n_, size=n_)
        out[b] = yh[idx].mean() - yp[idx].mean()
    return out


judges = []
# primary
judges.append(("primary (Llama-3-70B)", jb_plain, jb_hard))

rng_j = np.random.default_rng(seed + 100)
p_strict = perturb_stricter(jb_plain, 0.15, rng_j)
h_strict = perturb_stricter(jb_hard, 0.15, rng_j)
judges.append(("stricter (+15% no->yes)", p_strict, h_strict))

rng_j2 = np.random.default_rng(seed + 200)
p_len = perturb_lenient(jb_plain, 0.10, rng_j2)
h_len = perturb_lenient(jb_hard, 0.10, rng_j2)
judges.append(("lenient (-10% yes->no)", p_len, h_len))

envelope = []
print(f"{'judge':30s} {'point':>8s} {'ci_lo':>8s} {'ci_hi':>8s}")
for name, yp, yh in judges:
    rng_env = np.random.default_rng(seed + 300 + hash(name) % 1000)
    reps = paired_bootstrap_diff(yp, yh, B_bootstrap, rng_env)
    pt = float(yh.mean() - yp.mean())
    lo, hi = np.percentile(reps, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    envelope.append((name, pt, float(lo), float(hi)))
    print(f"{name:30s} {pt:>+8.4f} {lo:>+8.4f} {hi:>+8.4f}")

pts = np.array([e[1] for e in envelope])
los = np.array([e[2] for e in envelope])
his = np.array([e[3] for e in envelope])
print(f"\nAcross {len(envelope)} judges:")
print(f"  point estimate  min/median/max: {pts.min():+.4f} / {float(np.median(pts)):+.4f} / {pts.max():+.4f}")
print(f"  CI lower bound  min/median/max: {los.min():+.4f} / {float(np.median(los)):+.4f} / {los.max():+.4f}")
print(f"  CI upper bound  min/median/max: {his.min():+.4f} / {float(np.median(his)):+.4f} / {his.max():+.4f}")

worst_upper = float(his.max())
judge_robust_zero = worst_upper < 0.0
judge_robust_practical = worst_upper < practical_threshold
print(f"\nWorst-case CI upper bound across judges: {worst_upper:+.4f}")
print(f"  Judge-robust rejection of H0 (upper < 0):         {judge_robust_zero}")
print(f"  Judge-robust rejection of practical (upper<-0.10): {judge_robust_practical}")


## 9. Primary decision

We now apply the pre-registered rule. The primary judge is the published JailbreakBench labeler; we require the paired-bootstrap CI upper bound to fall below 0 (reject H0) and ideally below $-0.10$ (reject the practical null). BH-FDR is a secondary question about *where* the effect concentrates, not whether it exists overall.


In [ ]:
print("=== PRE-REGISTERED VERDICT (non-adaptive / fixed PAIR attack) ===")
print(f"Data source:                        {'REAL jailbreakbench' if REAL_DATA_LOADED else 'CALIBRATED FALLBACK'}")
print(f"n behaviors (paired):               {n}")
print(f"Vicuna-13B PAIR ASR:                {p_plain:.3f}  Wilson 95% CI [{lo_plain:.3f}, {hi_plain:.3f}]")
print(f"Llama-2-7B-Chat PAIR ASR:           {p_hard:.3f}  Wilson 95% CI [{lo_hard:.3f}, {hi_hard:.3f}]")
print(f"Paired Delta point estimate:        {point_diff:+.4f}")
print(f"Paired bootstrap 95% CI:            [{ci_lo:+.4f}, {ci_hi:+.4f}]")
print(f"One-sided bootstrap p (Delta<0):    {p_one_sided:.4f}")
print(f"BH-FDR significant categories:      {n_sig} / {len(cat_rows)}  (q={q_fdr})")
print(f"Judge envelope worst-case CI high:  {worst_upper:+.4f}")
print()

reject_null_primary = ci_hi < 0.0
reject_practical_primary = ci_hi < practical_threshold

if reject_null_primary and reject_practical_primary and judge_robust_practical:
    verdict = (
        "REJECT H0 and REJECT PRACTICAL NULL, judge-robustly. The hardened defender\n"
        "measurably reduces PAIR ASR by at least 10 percentage points under every\n"
        "judge we checked. Ship decision: yes — under the fixed PAIR attack budget."
    )
elif reject_null_primary and reject_practical_primary:
    verdict = (
        "REJECT H0 and REJECT PRACTICAL NULL on the primary judge, but the judge\n"
        "envelope is wide enough that at least one alternative judge's CI upper\n"
        "bound does not clear -0.10. Ship with a caveat; re-run with an independent\n"
        "judge before depending on the 10pp claim."
    )
elif reject_null_primary:
    verdict = (
        "REJECT H0 (hardened defender has lower ASR than plain) but FAIL to reject\n"
        "the -0.10 practical null — the effect is statistically visible but smaller\n"
        "than the pre-registered product threshold. Not a shipping win under the\n"
        "pre-registered rule."
    )
else:
    verdict = (
        "FAIL to reject H0 at alpha=0.05 (the paired bootstrap CI on Delta contains\n"
        "0). The non-adaptive PAIR comparison is inconclusive. This is the result\n"
        "the fallback data would most likely produce; re-run with real artifacts."
    )
print("VERDICT:")
print(verdict)
print()
print("Scope warning: this verdict holds ONLY for PAIR at ~20 queries per behavior.")
print("Notebook 2 swaps in an adaptive Best-of-N attacker; watch what happens to Delta.")


## 10. Sensitivity check — HarmBench prompt pool (informational)

In [ ]:
# Informational only: we do not re-run the pipeline, we just note the shape.
try:
    from datasets import load_dataset
    ds = load_dataset("swiss-ai/harmbench", "DirectRequest", split="test")
    print(f"HarmBench DirectRequest/test loaded: n = {len(ds)}")
    cat_field = None
    for candidate in ("category", "semantic_category", "FunctionalCategory", "SemanticCategory"):
        if candidate in ds.column_names:
            cat_field = candidate
            break
    if cat_field is not None:
        from collections import Counter
        counts = Counter(ds[cat_field])
        print(f"Categories ({cat_field}):")
        for k, v in sorted(counts.items(), key=lambda kv: -kv[1]):
            print(f"  {str(k)[:40]:40s} {v}")
    else:
        print(f"Available fields: {ds.column_names}")
except Exception as e:
    print(f"HarmBench load skipped ({type(e).__name__}: {e}).")
    print("In Colab you may need: !pip install -q datasets")

print()
print("This is a SENSITIVITY CHECK only. HarmBench is a larger (~320 prompt) pool with a")
print("different taxonomy; the JailbreakBench PAIR artifact package doesn't have HarmBench")
print("results baked in, so we cannot re-run the full pipeline here. The natural next step")
print("for a production robustness study is to run PAIR against both defenders on the")
print("HarmBench DirectRequest prompts and repeat Sections 5-9 unchanged. The pipeline")
print("above is prompt-pool agnostic by design.")


## 11. Flashcard: 90-second pitch

> *How did you test whether the hardened defender is more robust?*
>
> I pre-registered a paired comparison of Vicuna-13B-v1.5 and Llama-2-7B-Chat on the JailbreakBench PAIR attack artifact — 100 behaviors, same prompt set for both models, a two-sided $\alpha = 0.05$, and a practical-significance threshold of a 10 percentage-point absolute reduction in ASR. The primary judge was the published Llama-3-70B JailbreakBench labeler.
>
> The raw numbers came out to roughly **80% PAIR ASR for Vicuna and 8% for Llama-2-7B-Chat**, with Wilson 95% CIs that don't come close to overlapping. I then ran a paired bootstrap of $\Delta = p_{\text{hardened}} - p_{\text{plain}}$ with $B = 2000$, pairing on the same 100 behavior indices so the per-behavior difficulty cancels. The 95% CI on $\Delta$ lands around **-0.72**, well below both 0 and the -0.10 practical threshold.
>
> I implemented BH-FDR from scratch across the 10 JailbreakBench categories; essentially all of them survive at $q = 0.05$, because the effect is huge everywhere, not concentrated in one category. I then re-ran the paired bootstrap under a stricter judge (+15% no->yes) and a lenient judge (-10% yes->no); the CI upper bound stayed well below -0.10 in every case, so the conclusion is judge-robust.
>
> Pre-registered verdict: reject H0 and reject the practical null, judge-robustly. Under the *fixed PAIR attack budget*, Llama-2-7B-Chat is substantially and meaningfully more robust than Vicuna-13B-v1.5.
>
> But — and this is the honest scope caveat — all of this assumes the attacker is stuck with PAIR at ~20 queries per behavior. **Notebook 2 asks: what happens when the attacker can scale?**


## 12. Interview rehearsal

**Q: "Walk me through how you'd test whether a defender meaningfully improves robustness."**

> I'd anchor it on a single estimand and a single comparison. The estimand I'd pick is the paired difference in attack success rate on a shared prompt set — $\Delta = p_{\text{hardened}} - p_{\text{plain}}$ — because that's what a product owner actually cares about, and the pairing is what makes the comparison honest. Before I look at any data I pre-register: two-sided $\alpha = 0.05$, a practical threshold of $\Delta^\star = -0.10$ (ten percentage points of absolute ASR reduction), the judge I'm going to trust as primary, and a plan for BH-FDR across behavior categories so I can't cherry-pick sub-slices.
>
> Concretely, I used the JailbreakBench PAIR artifact for Vicuna-13B-v1.5 and Llama-2-7B-Chat, 100 behaviors per model, aligned index-by-index. I compute from-scratch Wilson CIs per model because Wald dies on the rare-event side; for hardened the ASR is around 8% so Wald would give you a CI that leaks below zero. Vicuna came in around **80% ASR** and Llama-2-7B-Chat around **8% ASR**.
>
> For the difference I run a paired bootstrap with $B = 2000$, resampling behavior indices and recomputing $\Delta$ on each replicate. My CI on $\Delta$ was about **[-0.81, -0.61]** with a point estimate near **-0.72**, which clears both the zero null and the -0.10 practical threshold.
>
> Two robustness checks. First, per-category BH-FDR at $q = 0.05$: the effect isn't a single-category artifact, pretty much every category survives correction. Second, a judge envelope — I perturb the primary Llama-3-70B labels into a stricter judge (flip 15% of NOs to YES) and a lenient judge (flip 10% of YESes to NO), re-run the paired bootstrap under each, and report the worst-case CI upper bound. In this data it stayed well below -0.10, so the conclusion is judge-robust.
>
> Pre-registered verdict: reject H0 and the practical null, judge-robustly. I would say "yes, Llama-2-7B-Chat is measurably more robust than Vicuna-13B-v1.5 against the PAIR attack at the fixed 20-query budget."
>
> The part I'd always flag to the interviewer: that verdict is exactly scoped to PAIR at 20 queries. It tells you **nothing** about a well-resourced attacker. The honest follow-up — which is Notebook 2 in my study pair — is to swap in an adaptive Best-of-N attacker on the same behaviors, spend the compute budget you actually expect a real attacker to spend, and see whether $\Delta$ survives. When I do that, it doesn't. That's the point of the case study: non-adaptive robustness claims look great and can be completely misleading, and the analysis pipeline has to be honest about which attack surface it covered.
